# 13 — Confidence Scheduling

**Engineering statement:** confidence specifies verification allocation.

Notebook 13 is the flagship algorithm notebook for `confidence-scheduled-decoding`. Notebook 00 introduced the decoding bottleneck. Notebook 07 reframed verification as a scarce engineering resource. Notebook 13 asks how confidence schedules that resource.

The goal is not to reproduce the full DSpark serving stack. The goal is to build a small executable model of confidence scores, prefix survival, verification allocation, and expected throughput.

## Start Here

```text
00: decoding creates latency pressure
07: verification becomes an engineering resource
13: confidence schedules that resource
```

The notebook flow is:

```text
confidence scores
→ prefix survival
→ scheduling policy
→ verification allocation
→ accepted prefix
→ throughput
```

In [ ]:
from pathlib import Path
import json
import math
import shutil
import sys
import zipfile
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Robust paths whether run from repo root, notebooks/, or a copied notebook.
CWD = Path.cwd().resolve()
if (CWD / "src").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "13_confidence_scheduling"
SITE = REPO_ROOT / "site" / "2606-19348"
SITE_IMAGES = SITE / "images"

for p in [NOTEBOOKS, FIGURES, RESULTS, SITE_IMAGES]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")
print(f"Site img:  {SITE_IMAGES}")

## 1. Confidence becomes a scheduling variable

A confidence score is useful only when it changes an engineering decision. In DSpark-style decoding, the decision is not simply whether a token looks likely. The decision is how much target-model verification should be allocated to each draft prefix.

A single token may have high conditional confidence, but speculative verification accepts a **continuous prefix**. Therefore the scheduling variable is the cumulative probability that the whole prefix survives.

In [ ]:
@dataclass(frozen=True)
class RequestProfile:
    name: str
    confidences: list

profiles = [
    RequestProfile("chat",        [0.82, 0.74, 0.62, 0.48, 0.36, 0.25, 0.18, 0.12]),
    RequestProfile("instruction", [0.88, 0.82, 0.74, 0.63, 0.52, 0.40, 0.30, 0.22]),
    RequestProfile("code",        [0.96, 0.94, 0.91, 0.87, 0.80, 0.72, 0.62, 0.50]),
    RequestProfile("math",        [0.95, 0.92, 0.88, 0.82, 0.73, 0.63, 0.52, 0.40]),
]

def prefix_survival(conf):
    conf = np.asarray(conf, dtype=float)
    return np.cumprod(conf)

def expected_accepts(survival, ell=None, bonus=True):
    survival = np.asarray(survival, dtype=float)
    if ell is None:
        ell = len(survival)
    base = 1.0 if bonus else 0.0
    return base + survival[:ell].sum()

rows = []
for prof in profiles:
    surv = prefix_survival(prof.confidences)
    for j, (c, a) in enumerate(zip(prof.confidences, surv), start=1):
        rows.append({
            "request": prof.name,
            "position": j,
            "conditional_confidence_cj": c,
            "prefix_survival_aj": a,
        })
confidence_df = pd.DataFrame(rows)
confidence_path = RESULTS / "confidence_prefix_survival.csv"
confidence_df.to_csv(confidence_path, index=False)
confidence_df.head(12)

In [ ]:
# Figure 13_confidence_decay.png
positions = np.arange(1, 9)
example = profiles[0]
conf = np.asarray(example.confidences)
surv = prefix_survival(conf)

fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.plot(positions, conf, marker="o", label="conditional confidence $c_j$")
ax.plot(positions, surv, marker="o", label="prefix survival $a_j$")
ax.set_xlabel("Draft position j")
ax.set_ylabel("Probability")
ax.set_ylim(0, 1.05)
ax.set_title("Confidence becomes a prefix-level scheduling variable")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()

fig_path = FIGURES / "13_confidence_decay.png"
fig.savefig(fig_path, dpi=180)
shutil.copyfile(fig_path, SITE_IMAGES / fig_path.name)
plt.show()
fig_path

## 2. Prefix survival families

The core transformation is:

\[
a_j=\prod_{i=1}^{j}c_i.
\]

This turns conditional token confidence into a prefix survival curve. Different request types can produce very different curves. Structured code or math often sustains longer prefixes than open-ended chat, so a single fixed verification length is not a good universal policy.

In [ ]:
# Figure 13_prefix_survival_families.png
fig, ax = plt.subplots(figsize=(8.6, 5.0))
for prof in profiles:
    ax.plot(positions, prefix_survival(prof.confidences), marker="o", label=prof.name)
ax.set_xlabel("Draft position j")
ax.set_ylabel("Prefix survival $a_j$")
ax.set_ylim(0, 1.05)
ax.set_title("Prefix survival depends on request structure")
ax.grid(True, alpha=0.25)
ax.legend(title="request")
fig.tight_layout()

fig_path = FIGURES / "13_prefix_survival_families.png"
fig.savefig(fig_path, dpi=180)
shutil.copyfile(fig_path, SITE_IMAGES / fig_path.name)
plt.show()
fig_path

## 3. Scheduling policies

Once confidence is converted to prefix survival, several policies become possible.

```text
verify everything
→ fixed prefix
→ confidence threshold
→ budget-aware scheduling
```

The important shift is that verification length becomes a decision variable. Confidence does not merely annotate a draft; it determines how much of the draft enters target-model verification.

In [ ]:
def fixed_prefix_policy(survival, fixed_len=4):
    return min(fixed_len, len(survival))

def threshold_policy(survival, threshold=0.35):
    ell = int(np.sum(np.asarray(survival) >= threshold))
    return ell

def expected_waste(survival, ell):
    return ell - np.asarray(survival)[:ell].sum()

policy_rows = []
for prof in profiles:
    surv = prefix_survival(prof.confidences)
    policies = {
        "verify all": len(surv),
        "fixed prefix": fixed_prefix_policy(surv, 4),
        "threshold": threshold_policy(surv, 0.35),
    }
    for pol, ell in policies.items():
        policy_rows.append({
            "request": prof.name,
            "policy": pol,
            "scheduled_length": ell,
            "expected_accepts": expected_accepts(surv, ell),
            "expected_waste": expected_waste(surv, ell),
        })
policy_df = pd.DataFrame(policy_rows)
policy_path = RESULTS / "scheduler_policy_comparison.csv"
policy_df.to_csv(policy_path, index=False)
policy_df

In [ ]:
# Figure 13_scheduler_policies.png
pivot = policy_df.pivot(index="request", columns="policy", values="scheduled_length")
# stable column order
pivot = pivot[["verify all", "fixed prefix", "threshold"]]

fig, ax = plt.subplots(figsize=(8.8, 4.8))
x = np.arange(len(pivot.index))
width = 0.24
for i, col in enumerate(pivot.columns):
    ax.bar(x + (i-1)*width, pivot[col], width=width, label=col)
ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel("Scheduled verification length $\\ell$")
ax.set_title("Scheduling policies choose different verification lengths")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

fig_path = FIGURES / "13_scheduler_policies.png"
fig.savefig(fig_path, dpi=180)
shutil.copyfile(fig_path, SITE_IMAGES / fig_path.name)
plt.show()
fig_path

## 4. Verification budget allocation

A scheduler is most useful when multiple active requests compete for target-model capacity. The allocation question is:

> Which prefixes should receive verification tokens first?

A simple budget-aware policy ranks candidate prefix extensions by survival probability and admits the highest-value extensions until the verification budget is exhausted.

In [ ]:
def budget_allocate(profile_list, token_budget):
    """Allocate a fixed number of draft verification tokens across requests.

    Prefix constraints are respected by ranking all prefix extensions by survival.
    Because survival is monotonically non-increasing within a request, selecting the
    top extensions yields contiguous prefixes in this illustrative setting.
    """
    candidates = []
    survival_by_req = {}
    for prof in profile_list:
        surv = prefix_survival(prof.confidences)
        survival_by_req[prof.name] = surv
        for j, a in enumerate(surv, start=1):
            candidates.append((prof.name, j, float(a)))
    candidates.sort(key=lambda x: x[2], reverse=True)
    selected = candidates[:token_budget]
    lengths = {prof.name: 0 for prof in profile_list}
    for name, j, a in selected:
        lengths[name] = max(lengths[name], j)
    return lengths, selected, survival_by_req

budget = 14
lengths, selected, survival_by_req = budget_allocate(profiles, budget)
allocation_df = pd.DataFrame([
    {
        "request": name,
        "scheduled_length": ell,
        "expected_accepts_without_bonus": survival_by_req[name][:ell].sum(),
        "expected_waste": expected_waste(survival_by_req[name], ell),
    }
    for name, ell in lengths.items()
])
allocation_path = RESULTS / "budget_allocation.csv"
allocation_df.to_csv(allocation_path, index=False)
allocation_df

In [ ]:
# Figure 13_budget_allocator.png
fig, ax = plt.subplots(figsize=(9.2, 4.6))
requests = [p.name for p in profiles]
y = np.arange(len(requests))
max_len = 8
for i, name in enumerate(requests):
    ell = lengths[name]
    for j in range(1, max_len + 1):
        value = survival_by_req[name][j-1]
        alpha = 0.95 if j <= ell else 0.20
        ax.barh(i, 1, left=j-1, height=0.6, alpha=alpha, edgecolor="black")
        if j <= ell:
            ax.text(j-0.5, i, f"{value:.2f}", ha="center", va="center", fontsize=8)
ax.set_yticks(y)
ax.set_yticklabels(requests)
ax.set_xlim(0, max_len)
ax.set_xlabel("Draft position")
ax.set_title("Budget-aware scheduler allocates verification to high-survival prefixes")
ax.invert_yaxis()
fig.tight_layout()

fig_path = FIGURES / "13_budget_allocator.png"
fig.savefig(fig_path, dpi=180)
shutil.copyfile(fig_path, SITE_IMAGES / fig_path.name)
plt.show()
fig_path

## 5. Throughput optimization

Expected accepted tokens alone are not enough. Verifying more tokens can increase expected accepts while also increasing batch size and reducing steps per second.

Notebook 13 uses a deliberately simple throughput curve:

\[
\Theta(\ell)=\tau(\ell)\,\mathrm{SPS}(B(\ell)).
\]

The optimum occurs where additional expected accepts no longer compensate for the cost of extra verification load.

In [ ]:
def sps(batch_size, base=1.0, slope=0.035, curvature=0.0009):
    """Illustrative steps/sec profile. Larger verification batches reduce step rate."""
    b = np.asarray(batch_size, dtype=float)
    return base / (1.0 + slope * b + curvature * b**2)

def throughput_for_profile(prof, max_len=8, active_requests=48):
    surv = prefix_survival(prof.confidences)
    rows = []
    for ell in range(0, max_len + 1):
        tau = expected_accepts(surv, ell)
        B = active_requests * (1 + ell)
        theta = tau * sps(B)
        rows.append({"request": prof.name, "ell": ell, "tau": tau, "B": B, "SPS": sps(B), "Theta": theta})
    return pd.DataFrame(rows)

throughput_df = pd.concat([throughput_for_profile(p) for p in profiles], ignore_index=True)
throughput_path = RESULTS / "throughput_by_scheduled_length.csv"
throughput_df.to_csv(throughput_path, index=False)
throughput_df.head()

In [ ]:
# Figure 13_throughput_surface.png
fig, ax = plt.subplots(figsize=(8.8, 5.0))
for name, group in throughput_df.groupby("request"):
    ax.plot(group["ell"], group["Theta"], marker="o", label=name)
    best = group.loc[group["Theta"].idxmax()]
    ax.scatter([best["ell"]], [best["Theta"]], s=90, zorder=5)
ax.set_xlabel("Scheduled verification length $\\ell$")
ax.set_ylabel("Expected throughput $\\Theta$")
ax.set_title("Throughput has a request-dependent scheduling optimum")
ax.grid(True, alpha=0.25)
ax.legend(title="request")
fig.tight_layout()

fig_path = FIGURES / "13_throughput_surface.png"
fig.savefig(fig_path, dpi=180)
shutil.copyfile(fig_path, SITE_IMAGES / fig_path.name)
plt.show()
fig_path

## 6. Greedy admission path

The DSpark scheduler ranks candidate prefix extensions by expected survival value, admits high-value verification tokens, updates expected throughput, and stops when the next admission no longer improves the objective.

```text
rank candidate prefix extensions
→ admit high-survival tokens
→ update expected throughput
→ stop when throughput falls
```

This is the operational meaning of confidence scheduling: confidence produces a priority order for spending verification budget.

In [ ]:
def greedy_admission(profile_list, max_candidates=None, active_requests=48):
    candidates = []
    for prof in profile_list:
        surv = prefix_survival(prof.confidences)
        for j, a in enumerate(surv, start=1):
            candidates.append((prof.name, j, float(a)))
    candidates.sort(key=lambda x: x[2], reverse=True)
    if max_candidates is not None:
        candidates = candidates[:max_candidates]

    lengths = {p.name: 0 for p in profile_list}
    expected_accepts_total = len(profile_list)  # one target bonus/anchor per request in this illustrative accounting
    rows = [{"admitted": 0, "last_request": "baseline", "last_position": 0, "gain": 0.0,
             "batch_size": active_requests, "expected_accepts": expected_accepts_total,
             "throughput": expected_accepts_total * sps(active_requests)}]
    for k, (name, j, gain) in enumerate(candidates, start=1):
        # Prefix-respecting update: by construction j arrives after earlier same-request extensions.
        lengths[name] = max(lengths[name], j)
        expected_accepts_total += gain
        batch = active_requests + k
        theta = expected_accepts_total * sps(batch)
        rows.append({"admitted": k, "last_request": name, "last_position": j, "gain": gain,
                     "batch_size": batch, "expected_accepts": expected_accepts_total,
                     "throughput": theta})
    return pd.DataFrame(rows)

greedy_df = greedy_admission(profiles, active_requests=48)
greedy_path = RESULTS / "greedy_admission_path.csv"
greedy_df.to_csv(greedy_path, index=False)
greedy_df.head(12)

In [ ]:
# Figure 13_greedy_admission_path.png
fig, ax = plt.subplots(figsize=(9.0, 4.8))
ax.plot(greedy_df["admitted"], greedy_df["throughput"], marker="o")
best = greedy_df.loc[greedy_df["throughput"].idxmax()]
ax.scatter([best["admitted"]], [best["throughput"]], s=110, zorder=5)
ax.annotate(
    f"stop near optimum\n{int(best['admitted'])} admitted tokens",
    xy=(best["admitted"], best["throughput"]),
    xytext=(best["admitted"] + 2, best["throughput"] * 0.98),
    arrowprops=dict(arrowstyle="->", linewidth=1.4),
)
ax.set_xlabel("Admitted verification tokens")
ax.set_ylabel("Expected throughput")
ax.set_title("Greedy admission path for confidence-scheduled verification")
ax.grid(True, alpha=0.25)
fig.tight_layout()

fig_path = FIGURES / "13_greedy_admission_path.png"
fig.savefig(fig_path, dpi=180)
shutil.copyfile(fig_path, SITE_IMAGES / fig_path.name)
plt.show()
fig_path

In [ ]:
# Figure 13_scheduler_flow.png
steps = [
    "confidence\nscores",
    "prefix\nsurvival",
    "rank prefix\nextensions",
    "admit high\nvalue tokens",
    "update\nthroughput",
    "scheduled\nprefixes",
]
fig, ax = plt.subplots(figsize=(11, 2.8))
ax.axis("off")
xs = np.linspace(0.07, 0.93, len(steps))
y = 0.5
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(x, y, label, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes)
    if i < len(steps)-1:
        ax.annotate("", xy=(xs[i+1]-0.06, y), xytext=(x+0.06, y), xycoords=ax.transAxes,
                    arrowprops=dict(arrowstyle="->", lw=1.5))
ax.set_title("Confidence scheduling flow", fontsize=15)
fig.tight_layout()

fig_path = FIGURES / "13_scheduler_flow.png"
fig.savefig(fig_path, dpi=180)
shutil.copyfile(fig_path, SITE_IMAGES / fig_path.name)
plt.show()
fig_path

## 7. Transition to semi-autoregressive drafting

Notebook 13 explains how confidence allocates verification. Notebook 17 asks where better confidence and longer useful prefixes come from.

```text
confidence scheduling
→ semi-autoregressive generation
→ better suffix quality
→ longer useful prefixes
→ higher throughput
```

In [ ]:
# Figure 13_transition.png
steps = ["confidence\nscheduling", "semi-autoregressive\ngeneration", "better suffix\nquality", "longer useful\nprefixes", "higher\nthroughput"]
fig, ax = plt.subplots(figsize=(10, 3.0))
ax.axis("off")
xs = np.linspace(0.08, 0.92, len(steps))
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(x, 0.5, label, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes)
    if i < len(steps)-1:
        ax.annotate("", xy=(xs[i+1]-0.07, 0.5), xytext=(x+0.07, 0.5), xycoords=ax.transAxes,
                    arrowprops=dict(arrowstyle="->", lw=1.5))
ax.set_title("Notebook transition: scheduling needs better schedulable prefixes", fontsize=15)
fig.tight_layout()

fig_path = FIGURES / "13_transition.png"
fig.savefig(fig_path, dpi=180)
shutil.copyfile(fig_path, SITE_IMAGES / fig_path.name)
plt.show()
fig_path

## 8. Engineering summary

Confidence scheduling turns a set of draft-token probabilities into an allocation policy.

| Engineering object | Scheduling role |
|---|---|
| conditional confidence \(c_j\) | local token-quality estimate |
| prefix survival \(a_j\) | expected marginal value of verifying position \(j\) |
| verification length \(\ell\) | per-request resource allocation |
| batch size \(B\) | system-level cost of verification |
| throughput \(\Theta\) | objective for useful serving work |

The result is a clean engineering transition:

```text
confidence estimate
→ prefix value
→ verification allocation
→ throughput objective
```

## 9. Key equations

Prefix survival:

\[
a_j=\prod_{i\le j}c_i.
\]

Expected accepted length:

\[
\tau(\ell)=1+\sum_{j=1}^{\ell}a_j.
\]

Verification batch size:

\[
B=\sum_{r=1}^{R}(1+\ell_r).
\]

Hardware-aware throughput objective:

\[
\Theta=\tau\cdot \mathrm{SPS}(B).
\]

Scheduled verification length:

\[
\ell^*=\arg\max_{\ell}\Theta(\ell).
\]

In [ ]:
roadmap = pd.DataFrame([
    {"notebook": "00_context", "specification": "decoding pressure", "question": "Why is decoding expensive?"},
    {"notebook": "07_verification_resource", "specification": "verification resource", "question": "Why does verification need scheduling?"},
    {"notebook": "13_confidence_scheduling", "specification": "confidence allocation", "question": "How does confidence schedule verification?"},
    {"notebook": "17_semi_autoregressive_design", "specification": "schedulable prefixes", "question": "How does the drafter improve suffix quality?"},
    {"notebook": "23_throughput_objective", "specification": "throughput optimization", "question": "Where is the optimum?"},
    {"notebook": "29_hardware_aware_scheduling", "specification": "engine-aware allocation", "question": "How does hardware load change the policy?"},
    {"notebook": "37_operating_regimes", "specification": "load regimes", "question": "When should the system verify more or less?"},
    {"notebook": "43_resource_allocation", "specification": "adaptive compute", "question": "How does this generalize beyond decoding?"},
    {"notebook": "49_adaptive_ai_infrastructure", "specification": "AI operating systems", "question": "What does adaptive inference infrastructure require?"},
])
roadmap_path = RESULTS / "notebook_roadmap.csv"
roadmap.to_csv(roadmap_path, index=False)
roadmap

In [ ]:
references = {
    "paper": {
        "identifier": "2606.19348",
        "title": "DSpark: Confidence-Scheduled Speculative Decoding with Semi-Autoregressive Generation",
        "url": "https://arxiv.org/html/2606.19348v1",
    },
    "github": "https://github.com/deepseek-ai/DeepSpec",
    "model": "https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro-DSpark",
    "repo": "https://github.com/thinkthoughts/confidence-scheduled-decoding",
}
references_path = RESULTS / "references.json"
references_path.write_text(json.dumps(references, indent=2), encoding="utf-8")
references

## 10. Download artifacts

Run the final cell to package Notebook 13 outputs for download.

In [ ]:
from IPython.display import FileLink, display

notebook_manifest = {
    "notebook": "13_confidence_scheduling.ipynb",
    "title": "Confidence Scheduling",
    "engineering_statement": "Confidence specifies verification allocation.",
    "outputs": {
        "confidence_prefix_survival_csv": str(confidence_path.relative_to(REPO_ROOT)),
        "scheduler_policy_comparison_csv": str(policy_path.relative_to(REPO_ROOT)),
        "budget_allocation_csv": str(allocation_path.relative_to(REPO_ROOT)),
        "throughput_by_scheduled_length_csv": str(throughput_path.relative_to(REPO_ROOT)),
        "greedy_admission_path_csv": str(greedy_path.relative_to(REPO_ROOT)),
        "roadmap_csv": str(roadmap_path.relative_to(REPO_ROOT)),
        "references_json": str(references_path.relative_to(REPO_ROOT)),
        "figures": [
            "figures/13_confidence_decay.png",
            "figures/13_prefix_survival_families.png",
            "figures/13_scheduler_policies.png",
            "figures/13_budget_allocator.png",
            "figures/13_throughput_surface.png",
            "figures/13_greedy_admission_path.png",
            "figures/13_scheduler_flow.png",
            "figures/13_transition.png",
        ],
    },
}
manifest_path = RESULTS / "13_confidence_scheduling_manifest.json"
manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")

zip_path = RESULTS / "13_confidence_scheduling_artifacts.zip"
paths_to_include = [RESULTS, FIGURES]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        for path in Path(item).rglob("*"):
            if not path.is_file():
                continue
            if path.resolve() == zip_path.resolve():
                continue
            try:
                archive_name = path.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = path.name
            zf.write(path, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass